# 微调生成模型

在本章中，我们将对一个预训练文本生成模型进行微调。微调对于生成高质量的模型至关重要，也是我们让模型适应特定预期行为的重要工具。通过微调，我们可以使模型适配特定数据集或领域。

本章将介绍微调文本生成模型的两种最常见的方法：监督微调(supervised fine-tuning)和偏好调优(preference tuning)。我们将探索微调预训练文本生成模型的变革性潜力，使其成为对应用更有效的工具。

## 12.1 LLM训练三步走：预训练、监督微调和偏好调优

创建高质量的LLM通常包括三个步骤。

#### 语言建模（预训练）

要创建高质量的LLM，第一步是在一个或多个大规模文本数据集上对模型进行预训练（见图12-1）​。在训练过程中，模型尝试预测下一个词元，以准确学习文本中的语言和语义表示。正如我们在第3章和第11章中所看到的，这称为语言建模，是一种自监督学习方法。

这一步会生成一个基座模型，通常也称为预训练模型或基础模型。基座模型是训练过程的关键产物，但对最终用户来说较难直接使用。这就是为什么下一步非常重要。

<div align="center">
    <img src="./picture/12-1.jpg">
    <p>图12-1：在语言建模过程中，LLM旨在根据输入预测下一个词元。这是一个无数据标注的过程</p>
</div>


#### 第一次微调（监督微调）

LLM如果能够在响应和遵循指令方面做得更好，就会更实用。当我们要求模型写一篇文章时，我们期望模型生成完整的文章，而不是列出与“写文章”类似的指令（基座模型可能这样做）​。

通过监督微调(SFT)，我们可以使基座模型学会遵循指令。在微调过程中，基座模型的参数会更新，以更好地适应目标任务，比如遵循指令。与预训练模型类似，它使用下一个词元的预测任务进行训练，但不是仅预测已有文章中的下一个词元，而是基于用户输入，预测应该回复的下一个词元（见图12-2）​。

<div align="center">
    <img src="./picture/12-2.jpg">
    <p>图12-2：在监督微调过程中，LLM旨在根据带附加标签的输入预测下一个词元。从某种意义上说，标签就是用户的输入</p>
</div>


监督微调也可用于其他任务（如分类）​，但通常用于将基座生成模型转变为指令（或对话）生成模型。

#### 第二次微调（偏好调优）

第二次微调的目的是进一步提高模型质量，使其更符合AI安全或人类偏好的预期行为。这称为偏好调优。偏好调优是微调的一种形式，顾名思义，它引导模型输出与我们的偏好保持一致，而偏好由我们提供的数据定义。与监督微调一样，偏好调优可以改进原始模型。此外，它还具有在训练过程中提炼输出偏好的优势。

图12-3展示了上述三个步骤，演示了从未训练的架构开始，到最终完成偏好调优LLM的过程。

<div align="center">
    <img src="./picture/12-3.jpg">
    <p>图12-3：创建高质量LLM的三个步骤</p>
</div>


在本章中，我们将使用一个已经在大规模数据集上训练好的基座模型，并探索如何通过两种微调策略对其进行微调。对于每种方法，我们都从理论基础开始，然后将其应用于实践。

## 12.2 监督微调

在大规模数据集上对模型进行预训练的目的是使其理解并生成符合语法和语义的语言。在这个过程中，模型学会补全输入短语，如图12-4所示。

<div align="center">
    <img src="./picture/12-4.jpg">
    <p>图12-4：基座LLM或预训练LLM被训练来预测下一个词</p>
</div>


这个例子也说明模型并未被训练来遵循指令，它会尝试补全问题而不是回答问题（见图12-5）​。

<div align="center">
    <img src="./picture/12-5.jpg">
    <p>图12-5：基座LLM不会遵循指令，而是尝试预测下一个词，甚至可能创建新的问题</p>
</div>


我们可以通过微调，使这个基座模型适配某些特定用例，比如遵循指令。

### 12.2.1 全量微调

最常见的微调方式是全量微调。与预训练LLM类似，全量微调过程涉及更新模型的所有参数，使其符合目标任务的要求。二者的主要区别在于，全量微调使用的是较小但已标注的数据集，而预训练是在没有任何标注的大型数据集上完成的（见图12-6）​。

<div align="center">
    <img src="./picture/12-6.jpg">
    <p>图12-6：与语言建模（预训练）相比，全量微调使用的是较小但已标注的数据集</p>
</div>


你可以使用任何标注数据进行全量微调，这种灵活性使其成为学习领域特定表示的绝佳技术。为了让LLM遵循指令，我们需要问答数据，这是一种指令数据。如图12-7所示，指令数据包含用户的指令和相应的答案。

<div align="center">
    <img src="./picture/12-7.jpg">
    <p>图12-7：指令数据包含用户的指令和相应的答案。这些指令可以包含多种任务</p>
</div>


在全量微调期间，模型接收输入（指令）并对输出（回复）进行下一个词元预测。这样，模型就不会生成新的问题，而是会遵循指令。

### 12.2.2 参数高效微调

更新模型的所有参数虽然可能显著提升模型的性能，但也有一些缺点，如训练成本高、训练时间长，并且需要大量存储空间等。为了解决这些问题，研究人员开始关注参数高效微调(parameter-efficient fine-tuning，PEFT)方法，这种方法旨在以更高的计算效率微调预训练模型。

#### 适配器

适配器(adapter)是许多基于PEFT的技术的核心组件。使用适配器的方案是，在Transformer内部引入一组额外的模块化组件，通过微调这些组件来提升模型在特定任务上的性能，而无须微调模型的所有权重。这节省了大量时间和计算资源。

适配器是在论文“Parameter-Efficient Transfer Learning forNLP”中提出的。该论文表明，仅微调BERT模型的3.6%的参数即可在特定任务上取得与微调全部模型权重相当的性能。​（这些参数通过适配器模块添加，并基于原始BERT模型进行了压缩处理。​）在GLUE基准测试中，论文展示的性能与全量微调的性能差距不到0.4%。如图12-8所示，在单个Transformer块中，该论文提出的架构将适配器分别放置在自注意力层和前馈神经网络层之后。

<div align="center">
    <img src="./picture/12-8.jpg">
    <p>图12-8：适配器在网络的特定位置添加少量可以高效微调的权重，同时保持模型的大部分权重不变</p>
</div>


然而，仅仅改变一个Transformer块是不够的，事实上，这些适配器被添加到了模型的每个Transformer块中，如图12-9所示。

通过观察模型中所有适配器组件的分布，我们可以清晰地看到单个适配器的结构，如图12-10所示。每个适配器可以专注于不同的任务，例如适配器1可以专门用于医疗文本分类，而适配器2可以专门用于命名实体识别。你可以从AdapterHub下载领域专用的适配器。

<div align="center">
    <img src="./picture/12-9.jpg">
    <p>图12-9：适配器组件分布在模型的各个Transformer块中</p>
</div>


<div align="center">
    <img src="./picture/12-10.jpg">
    <p>图12-10：专门用于特定任务的适配器可以被替换到相同的架构中，前提是它们共享原始模型架构和权重</p>
</div>


论文“AdapterHub: A Framework for AdaptingTransformers”提出了AdapterHub这个中央仓库，可以分享适配器。早期的许多适配器专注于BERT架构。近期，适配器已被应用到文本生成Transformer中，比如论文“LLaMA-Adapter: Efficient Fine-tuning of Language Models withZero-init Attention”中有相关阐述。

#### 低秩适配

作为适配器的替代方案，低秩适配(low-rank adaptation，LoRA)被引入。当前，LoRA是一种应用广泛且有效的参数高效  微调技术。与适配器类似，LoRA也只需要更新少量参数。如图12-11所示，它创建了基座模型的一个小型子集来进行微调，而没有向模型添加新层。

<div align="center">
    <img src="./picture/12-11.jpg">
    <p>图12-11：LoRA只需要微调一小部分可以与基座LLM分开保存的参数</p>
</div>


与适配器类似，由于只需要更新基座模型的一小部分，这种子集方法使得微调速度更快。我们通过用较小的矩阵近似原始LLM中的大矩阵来创建这个参数子集。然后，我们可以微调这些较小的矩阵，以替代直接微调原始的大矩阵。下面以图12-12所示的10×10矩阵为例进行介绍。

<div align="center">
    <img src="./picture/12-12.jpg">
    <p>图12-12：LLM的一个主要瓶颈是其庞大的权重矩阵。仅其中一个权重矩阵就可能包含1.5亿个参数，而每个Transformer块都有自己的权重矩阵</p>
</div>


我们可以构造两个较小的矩阵，将它们相乘可以重构一个10×10的矩阵。这将显著提高效率，因为我们现在只需使用20（10加10）个权重，而不是100（10乘以10）个权重，如图12-13所示。

<div align="center">
    <img src="./picture/12-13.jpg">
    <p>图12-13：将一个较大的权重矩阵分解为两个较小的矩阵，可以得到一个压缩的低秩矩阵，能够更高效地进行微调</p>
</div>


在训练过程中，我们只需要更新这些较小的矩阵，而无须对全部权重进行更新。然后，将更新后的变化矩阵（较小的矩阵）与冻结的全部权重组合在一起，如图12-14所示。

<div align="center">
    <img src="./picture/12-14.jpg">
    <p>图12-14：与全量微调相比，LoRA在训练期间更新原始权重的小型表示</p>
</div>


你可能会怀疑，使用适配器或LoRA方法会导致性能下降。确实如此。那么这种效率与性能的权衡在什么情况下是有意义的呢？

“Intrinsic Dimensionality Explains the Effectiveness ofLanguage Model Fine-Tuning”等论文证明，语言模型“具有非常低的内在维度”(have a very low intrinsic dimension)。这意味着我们可以找到能够近似LLM中巨大矩阵的低秩表示。例如，像GPT-3这样具有1750亿个参数的模型，在其96个Transformer块中，每一个Transformer块的内部都有一个12288×12288的权重矩阵，这意味着每个Transformer块有1.5亿个参数。如果我们能成功地将该矩阵适配到秩为8，那么每个块只需要两个12288×8的权重矩阵，也就是9.8万个参数。正如之前引用的论文“LoRA: Low-Rank Adaptation ofLarge Language Models”所解释的那样，这在速度、存储和计算方面大幅降低了成本。

这种低秩表示具有非常大的灵活性，它允许我们选择基座模型的特定部分进行微调。例如，我们可以只微调每个Transformer块中的查询(query)权重矩阵和值(value)权重矩阵。

#### 压缩模型以实现（更）高效的训练

我们可以通过降低模型原始权重的内存需求来进一步提升LoRA的效率，然后将这些权重投影到较小的矩阵。LLM的权重是具有特定精度的数值，可以用float64或float32等位数来表示。如图12-15所示，如果我们减少表示数值的位数，结果的精度就会降低。然而，减少位数也意味着降低该模型的内存需求。

<div align="center">
    <img src="./picture/12-15.jpg">
    <p>图12-15：尝试用32位浮点数和16位浮点数表示圆周率。注意，当我们将位数减半时，精度会降低</p>
</div>


我们希望通过量化，在减少表示权重的位数的同时，仍然能够准确表示原始的权重值。然而，如图12-16所示，当直接将高精度值映射为低精度值时，多个高精度值可能被映射为相同的低精度值。

<div align="center">
    <img src="./picture/12-16.jpg">
    <p>图12-16：量化相近的权重会导致重建后的权重相同，使得它们难以区分</p>
</div>


QLoRA（LoRA的量化版本）的作者发现了一种方法，可以在高位数精度和低位数精度之间进行转换，同时不会与原始权重产生太大差异。

QLoRA使用分块量化的方法将某些高精度值块映射为低精度值。QLoRA并不是直接将高精度值映射为低精度值，而是创建额外的块来量化相似的权重。如图12-17所示，这样可以用低精度准确地表示这些值。

<div align="center">
    <img src="./picture/12-17.jpg">
    <p>图12-17：分块量化可以通过量化块，以低精度准确地表示权重</p>
</div>


神经网络的一个良好特性是其值通常在-1和1之间呈正态分布。这个特性允许我们根据权重的相对密度将原始权重分配到较低的位数，如图12-18所示。因为考虑了权重的相对频率，所以权重之间的映射更加高效。这也减少了离群点带来的问题。

<div align="center">
    <img src="./picture/12-18.jpg">
    <p>图12-18：使用分布感知块，可以防止相近的值被表示为相同的量化值</p>
</div>


结合分块量化，这种标准化过程能够实现用低精度值准确地表示高精度值，同时LLM的性能只会略微降低。因此，我们可以从16位浮点数表示转换到仅需4位标准化浮点数表示。4位标准化浮点数表示显著降低了LLM在训练过程中的内存需求。请注意，LLM的量化通常对推理也很有帮助，因为量化后的LLM体积更小，所以需要的显存更少。

还有一些更优雅的方法可以进一步优化这一过程，比如双重量化和分页优化器。你可以在前面提到的关于QLoRA的论文“QLoRA: Efficient Finetuning of Quantized LLMs”中了解更多的相关内容。博客文章“A Visual Guide toQuantization”是关于量化的完整指南，且高度可视化。

## 12.3 使用QLoRA进行指令微调

在了解了QLoRA的工作原理后，我们将这些知识付诸实践。在本节中，我们将使用QLoRA微调Llama的一个完全开源且规模较小的版本——TinyLlama，使其能够遵循指令。我们可以将这个模型视为基座模型或预训练模型，它经过了语言建模训练，但还不能遵循指令。

### 12.3.1 模板化指令数据

为了让LLM遵循指令，我们需要准备遵循对话模板的指令数据。如图12-19所示，这个对话模板能够区分LLM生成的内容和用户生成的内容。

<div align="center">
    <img src="./picture/12-19.jpg">
    <p>图12-19：我们在本章中使用的对话模板</p>
</div>


我们选择使用这个对话模板来展示所有示例，因为TinyLlama的对话版本使用了相同的格式。我们使用的数据是UltraChat数据集的一个小子集。这个数据集是原始UltraChat数据集的过滤版本，包含近20万条用户与LLM之间的对话。

首先创建一个函数format_prompt来确保对话遵循这个模板。

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
# 加载分词器以使用其对话模板
template_tokenizer = AutoTokenizer.from_pretrained(
    "..\models\TinyLlama-1.1B-Chat-v1.0"
)
def format_prompt(example):
    """利用TinyLlama使用的<|user|>模板格式化提示词"""
    # 格式化回答
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}
# 加载数据并利用TinyLlama使用的模板进行格式化
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft", cache_dir="data")
    .shuffle(seed=42)
    .select(range(3_000))
)
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

为了缩短训练时间，我们选择了包含3000个文档的子集。你可以扩大这个子集，以获得更准确的结果。

我们可以使用text列探索格式化后的提示词：

In [2]:
# 格式化后的提示词示例
print(dataset["text"][2576])

<|user|>
Given the text: Knock, knock. Who’s there? Hike.
Can you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?</s>
<|assistant|>
Sure! Knock, knock. Who's there? Hike. Hike who? Hike up your pants, it's cold outside!</s>
<|user|>
Can you tell me another knock-knock joke based on the same text material "Knock, knock. Who's there? Hike"?</s>
<|assistant|>
Of course! Knock, knock. Who's there? Hike. Hike who? Hike your way over here and let's go for a walk!</s>



### 12.3.2 模型量化

现在我们有了数据，可以开始加载模型了。我们将应用QLoRA中的Q（量化）​，并使用bitsandbytes包将预训练模型压缩为4位表示。

我们可以在BitsAndBytesConfig中定义量化方案。我们将遵循原始QLoRA论文中的步骤，用4位精度加载模型(load_in_4bit)，并使用标准化的浮点数表示(bnb_4bit_quant_type)和双重量化(bnb_4bit_use_double_quant)。

In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    BitsAndBytesConfig,
)
model_name = "..\models\TinyLlama-1.1B-intermediate-step-1431k-3T"
# 4位量化配置——QLoRA中的Q
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 用4位精度加载模型
    bnb_4bit_quant_type="nf4",  # 量化类型
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算数据类型
    bnb_4bit_use_double_quant=True,  # 应用嵌套量化
)
# 强制非量化层使用bf16（trl 对量化模型强制 LoRA 为 bf16）
base_config = AutoConfig.from_pretrained(model_name)
base_config.dtype = torch.bfloat16
# 在GPU上加载要训练的模型
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=base_config,
    device_map="auto",
    # 普通SFT可以忽略此设置
    quantization_config=bnb_config,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
# 加载Llama分词器
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
C:\Users\admin\AppData\Local\Temp\ipykernel_88056\1832941836.py:8: SyntaxWarning: invalid escape sequence '\m'
  model_name = "..\models\TinyLlama-1.1B-intermediate-step-1431k-3T"
W1004 23:12:42.273000 88056 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

这种量化方法可以在保持大部分原始权重精度的同时减小原始模型的大小。量化后，加载模型只需要约1 GB显存。相比之下，不进行量化则需要约4 GB显存。请注意，在微调过程中，显存需求会增加，因此仅有加载模型所需的约1 GB显存是不够的。

### 12.3.3 LoRA配置

下面需要使用peft库定义LoRA配置，这代表微调过程的超参数：

In [4]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# 准备LoRA配置
peft_config = LoraConfig(
    lora_alpha=128,  # LoRA缩放
    lora_dropout=0.1,  # LoRA层的dropout
    r=64,  # 秩
    bias="none",
    task_type="CAUSAL_LM",
    target_modules= # 目标层
     ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj", "down_proj"]
)
# 准备用于训练的模型
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

需要注意以下参数。

r

压缩矩阵的秩（回顾图12-13）​。增大这个值会使压缩矩阵变大，从而降低压缩率，进而提高模型的表示能力。该参数的值通常在4和64之间。

lora_alpha

控制添加到原始权重的变化量。本质上，它平衡了原始模型的知识与新任务的知识。经验法则是将该参数的值设置为r值的两倍。

target_modules

控制哪些层作为目标。LoRA过程可以选择忽略特定层，如特定的投影层。这可以加快训练速度，但会降低性能，反之亦然。

调整以上参数是很有价值的实验，可以帮助你直观地理解哪些值有效，哪些值无效。你可以在Sebastian Raschka的Aheadof AI中找到关于LoRA微调的更多实用技巧。

这个示例展示了一种高效的模型微调方式。如果你想进行全量微调，可以在加载模型时移除quantization_config参数，并跳过创建peft_config。这样，你就可以从使用QLoRA的指令微调转向全量微调。

### 12.3.4 训练配置

最后，我们需要像第11章那样配置训练参数：

In [5]:
from trl import SFTConfig
output_dir = "./results"
# 训练参数
training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512
)

需要注意以下参数。

num_train_epochs

训练轮次。较大的值往往会降低性能，我们通常倾向于将该参数设为较小的值。

learning_rate

决定每次权重更新迭代的步长。QLoRA的作者发现，对于较大的模型（参数量超过330亿个）​，增大该参数的值，效果更好。

lr_scheduler_type

基于余弦的调度器，用于动态调整学习率。它会从零开始线性增加学习率，直到达到设定值。之后，学习率会按照余弦函数的值递减。

optim

原始QLoRA论文中使用的分页优化器。

优化这些参数是一项困难的任务，没有既定的指导方案。需要通过实验来找出最适合特定的数据集、模型大小和目标任务的参数取值。

虽然本节描述的是指令微调，但我们也可以使用QLoRA来微调指令模型。例如，我们可以微调对话模型以生成特定的SQL代码，或创建符合特定格式的JSON输出。只要有可用的数据（包含适当的查询-回复对）​，QLoRA就是一个非常有效的技术，可以把现有的对话模型微调得更适合具体用例。

### 12.3.5 训练

所有的模型和参数都已经准备就绪，我们可以开始对模型进行微调了。首先加载SFTTrainer，然后直接运行trainer.train()：

In [6]:
from trl import SFTTrainer
# 设置监督微调参数
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_arguments,
)
# 训练模型
trainer.train()
# 保存QLoRA权重
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

Step,Training Loss
10,1.591752
20,1.452318
30,1.436335
40,1.474991
50,1.464218
60,1.376739
70,1.488391
80,1.442034
90,1.422244
100,1.395907


根据logging_steps参数的设置，训练过程中每10步就会打印一次损失值。如果你使用的是Google Colab提供的免费GPU（在撰写本书时是Tesla T4）​，训练可能需要一小时左右。你可以趁机休息一会儿！

### 12.3.6 合并权重

在训练完QLoRA权重后，我们还需要将它们与原始权重结合才能使用。我们用16位精度而不是量化后的4位精度重新加载模型，以合并权重。虽然分词器在训练过程中没有更新，但为了便于访问，我们将其与模型保存在同一个文件夹中。

In [7]:
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)
# 合并LoRA和基座模型
merged_model = model.merge_and_unload()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

合并适配器和基座模型后，我们可以使用之前定义的提示词模板：

In [8]:
from transformers import pipeline
# 使用预定义的提示词模板
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
# 运行我们的指令微调模型
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
Large Language Models (LLMs) are machine learning algorithms that can process texts and generate natural language responses. They are used in a variety of contexts, such as chatbots, natural language processing, and speech recognition.

LLMs are trained on vast amounts of text data, which includes text from multiple sources, including books, websites, and social media. They are then used to generate responses that are as accurate as human-generated responses, but with the benefit of being faster and more efficient.

One of the main advantages of LLMs is their ability to understand complex concepts and generate responses that are both engaging and relevant to the context. This is made possible by the use of deep learning techniques, which allow LLMs to learn from large amounts of data and adapt their responses to different contexts and situations.

Another advantage of LLMs is their ability to generate responses i

聚合权重后的输出表明，模型能够很好地遵循我们的指令，这是基座模型无法实现的。

## 12.4 评估生成模型

评估生成模型是一个重大挑战。生成模型被应用于许多不同的场景，这意味着依赖单一指标进行评判比较困难。与专用模型不同，解决数学问题能力超群的生成模型不一定在解决编程问题时也能表现不俗。

与此同时，评估这些模型至关重要，特别是在需要一致性的生产环境中。由于其概率性质，生成模型不一定能产生一致的输出，因此需要进行稳健的评估。

在本节中，我们将探讨一些常见的评估方法，但需要强调的是，目前没有金标准。没有一个指标能完美适用于所有场景。

### 12.4.1 词级指标

评估生成模型的一类常见指标是词级指标。词级评估的经典技术在词元（集合）层面比较参考数据集与生成的词元。常见的词级指标包括困惑度(perplexity)、ROUGE、BLEU和BERTScore。


值得一提的是困惑度，它用于衡量语言模型对文本的预测能力。给定输入文本，模型预测下一个词元的概率。困惑度的基本假设是，如果模型给下一个词元一个较高的概率，则其表现更好。换句话说，当面对写得很好的文档时，模型不应该感到“困惑”​。

如图12-20所示，当给出输入When a measure becomes a时，模型需要判断target作为下一个词的概率有多大。

<div align="center">
    <img src="./picture/12-20.jpg">
    <p>图12-20：预测下一个词是许多LLM的核心特征</p>
</div>


尽管困惑度和其他词级指标对于理解模型的置信度很有用，但它们并不是完美的评估方法。这些指标无法衡量生成文本的一致性、流畅度、创造力，甚至正确性。

### 12.4.2 基准测试

要评估生成模型在语言生成和理解任务上的表现，一种常见方法是使用广为人知的公共基准测试，如MMLU、GLUE、TruthfulQA、GSM8k和HellaSwag。我们可以通过这些基准测试了解关于基础语言理解以及复杂的分析解答（如数学问题）的更多信息。


除了自然语言任务，一些模型专注于其他领域，如编程。这些模型通常会在不同的基准测试上进行评估，例如HumanEval，该基准测试包含一些具有挑战性的编程任务供模型解决。生成模型的常见公共基准测试如表12-1所示。


表12-1：生成模型的常见公共基准测试

基准测试是了解模型在各种任务上的表现的好方法。然而，公共基准测试也有其缺点，为了得到最佳回复，模型可能会过拟合基准测试。此外，这些基准测试通常比较宽泛，可能无法覆盖非常具体的应用场景。最后，某些基准测试需要强大的GPU，且计算时间较长（历经数小时）​，这使得模型迭代比较困难。

### 12.4.3 排行榜

由于基准测试种类繁多，选择适合自己模型的基准测试并不容易。新模型发布时，为了展示其在各项任务上的表现，通常会在多个基准测试上接受评估。

包含多个基准测试的排行榜应运而生。一个常见的排行榜是Open LLM Leaderboard。在撰写本书时，该排行榜包括HellaSwag、MMLU、TruthfulQA和GSM8k等6个基准测试。排行榜上名列前茅的模型（假设它们没有对数据过拟合）通常被认为是“最佳”模型。然而，由于这些排行榜通常包含公开可用的基准测试，因此存在模型对排行榜数据过拟合的风险。

### 12.4.4 自动评估

评估生成模型的输出时，一个重要的指标是其文本的质量。例如，即使两个模型针对同一个问题给出了相同的正确答案，它们得出答案的方式也可能不同。我们通常不仅关注最终答案，还关注答案的构建过程。类似地，尽管两个摘要相似，但其中一个可能比另一个显著简短，这对于一个好的摘要来说往往很重要。

为了在最终答案的正确性之外对生成文本的质量进行评估，研究人员引入了LLM-as-a-judge，也就是让另一个LLM来评判待评估的LLM的质量。这种方法的一个有趣的变体是成对比较，即两个不同的LLM分别生成针对同一个问题的答案，随后由第三个LLM作为裁判来判定哪个更好。

因此，这种方法允许对开放式问题进行自动评估，其主要优势在于，随着LLM的优化，它们评判输出质量的能力也会提高。换句话说，这种评估方法会随着技术的发展而不断改进。

### 12.4.5 人工评估

尽管基准测试至关重要，但评估的金标准通常被认为是人工评估。即使一个LLM在广泛的基准测试中表现出色，在特定领域的任务中仍可能表现不佳。此外，基准测试无法完全反映人类偏好，前面讨论的所有方法都只是人类偏好的替代指标。

Chatbot Arena是基于人工评估技术的绝佳示例。在这个排行榜中有两个匿名的LLM与你互动。你提出的任何问题或提示词都会同时发送给这两个模型，然后你会收到它们的输出。之后，你可以决定更喜欢哪个输出。这个过程使得社区成员在不知道具体是哪些模型的情况下，对他们偏好的模型进行投票。只有在你投票之后，你才能看到是哪个模型生成了哪段文本。

在撰写本书时，这种方法已经收集了80多万张人工投票，用于生成一个排行榜。基于这些投票，计算出胜率，也就是LLM的相对能力水平。如果一个排名靠后的LLM击败了一个排名靠前的LLM，其排名就会发生显著变化。在国际象棋中，这被称为Elo评分系统。

这种方法使用众包投票的方式来帮助我们了解LLM的质量。但是，它仍然只是综合了众多不同用户的意见，可能与我们的实际应用场景并不相关。

因此，目前还没有一种评估LLM的完美方法。前面提到的方法和基准测试都提供了重要但有限的评估视角。我们建议根据预期应用场景来评估LLM。比如对于编程来说，HumanEval比GSM8k更合适。

最重要的是，我们相信你才是最好的评估者。人工评估之所以是金标准，是因为最终由你决定LLM是否适合你的预期应用场景。就像本章的示例一样，我们强烈建议你试用这些模型，或许还可以自己设计一些问题。例如，当本书的作者遇到新模型时，经常用母语（Jay Alammar使用阿拉伯语，MaartenGrootendorst使用荷兰语）提问。

关于这个话题，最后分享一句我们珍视的名言：当一个指标成为目标时，它就不再是一个好的指标。——古德哈特定律

在LLM的语境下，当使用特定的基准测试时，我们往往会不顾后果地优化该基准测试。例如，当我们仅仅专注于优化生成语法正确的句子时，模型可能只学会输出一个句子：​“这是一个句子。​”这在语法上是正确的，但无法反映其语言理解能力。因此，模型可能在特定的基准测试上表现出色，但会牺牲其他有用的能力。

## 12.5 偏好调优、对齐

 尽管模型现在已经能够遵循指令，但我们可以通过最后的训练来进一步改进其行为，使其与我们期望它在不同场景中的表现保持一致。例如，当我们问“什么是LLM”时，我们可能更倾向于得到一个详细描述LLM内部机制的答案，而不是“它是一个LLM”​，不做进一步解释。那么，究竟如何将我们（人类）对一个答案优于另一个答案的偏好与LLM的输出对齐呢？

首先，回想一下LLM接收输入提示词并输出生成内容的情景，如图12-21所示。

<div align="center">
    <img src="./picture/12-21.jpg">
    <p>图12-21：LLM接收输入提示词并输出生成内容</p>
</div>


我们可以让一个人（偏好评估者）评估模型生成内容的质量。假设他给出一个特定分数，比如4（见图12-22）​。

<div align="center">
    <img src="./picture/12-22.jpg">
    <p>图12-22：让一个偏好评估者（人或其他方式）评估生成内容的质量</p>
</div>


图12-23展示了基于该分数更新模型的偏好调优步骤。

· 如果分数较高，就更新模型，以鼓励它产生更多这类生成内容。

· 如果分数较低，就更新模型，以抑制它产生这类生成内容。

<div align="center">
    <img src="./picture/12-23.jpg">
    <p>图12-23：偏好调优方法基于评估分数更新LLM</p>
</div>


和之前一样，我们需要大量训练样本。那么我们能否实现偏好评估自动化呢？是的，我们可以通过训练一个叫作奖励模型(reward model)的模型来实现。

## 12.6 使用奖励模型实现偏好评估自动化

要实现偏好评估自动化，我们需要在偏好调优步骤之前增加一个步骤，即训练一个奖励模型，如图12-24所示。

<div align="center">
    <img src="./picture/12-24.jpg">
    <p>图12-24：在微调LLM之前，我们先训练一个奖励模型</p>
</div>


如图12-25所示，要创建奖励模型，我们可以复制经过指令微调的模型，并稍作修改，使其不再生成文本，而是输出一个单一的分数。

<div align="center">
    <img src="./picture/12-25.jpg">
    <p>图12-25：通过将语言建模头替换为质量分类头，LLM变成了奖励模型</p>
</div>


### 12.6.1 奖励模型的输入和输出

奖励模型的预期工作方式是，我们给它一个提示词和一个生成内容，它会输出一个单一的数值，表示该生成内容对于该提示词的偏好/质量。图12-26展示了奖励模型生成这个单一数值的过程。

<div align="center">
    <img src="./picture/12-26.jpg">
    <p>图12-26：使用经过人类偏好训练的奖励模型生成补全内容的质量分数</p>
</div>


### 12.6.2 训练奖励模型


我们不能直接使用奖励模型。奖励模型需要先经过训练才能正确地对生成内容进行评分。因此，我们需要获取一个模型可以学习的偏好数据集。

#### 奖励模型训练数据集

偏好训练数据集的一种常见形式是，每个训练样本都包含一个提示词，以及一个被接受的生成内容和一个被拒绝的生成内容（注意，这并不总是“好”与“差”的对比，有时两个生成内容都很好，只是一个比另一个更好）​。图12-27展示了包含两个训练样本的偏好训练数据集示例。

<div align="center">
    <img src="./picture/12-27.jpg">
    <p>图12-27：偏好训练数据集通常由提示词及被接受和被拒绝的生成内容组成</p>
</div>


生成偏好数据的一种方法是向LLM提供一个提示词，让它生成两个不同的结果。如图12-28所示，我们可以请人工标注员选择他们更偏好哪一个。

<div align="center">
    <img src="./picture/12-28.jpg">
    <p>图12-28：输出两个生成内容并询问人工标注员更偏好哪一个</p>
</div>


#### 奖励模型训练步骤

有了偏好训练数据集之后，我们就可以开始训练奖励模型了。

一个简单的步骤是，我们使用奖励模型进行以下操作。

· 对被接受的生成内容评分。

· 对被拒绝的生成内容评分。

训练目标是确保被接受的生成内容的得分高于被拒绝的生成内容的得分，如图12-29所示。

<div align="center">
    <img src="./picture/12-29.jpg">
    <p>图12-29：奖励模型旨在评估针对提示词的生成内容的质量分数</p>
</div>


如图12-30所示，将上述步骤组合在一起，就得到了偏好调优的三个阶段。

· 收集偏好数据。

· 训练奖励模型。

· 微调LLM（奖励模型作为偏好评估器）​。

<div align="center">
    <img src="./picture/12-30.jpg">
    <p>图12-30：偏好调优的3个阶段：收集偏好数据、训练奖励模型、微调LLM</p>
</div>


奖励模型是一个很好的想法，可以进一步扩展和完善。例如，Llama 2训练了两个奖励模型：一个用于对有用性(helpfulness)评分，另一个用于对安全性(safety)评分（图12-31）​。

<div align="center">
    <img src="./picture/12-31.jpg">
    <p>图12-31：我们可以使用多个奖励模型来进行评分</p>
</div>


使用训练好的奖励模型来微调LLM的一种常用方法是近端策略优化(proximal policy optimization，PPO)。PPO是一种流行的强化学习技术，通过确保LLM不会过度偏离预期奖励来优化经过指令微调的LLM。2022年发布的最初版本的ChatGPT甚至也使用PPO进行训练。

### 12.6.3 训练无奖励模型

PPO的一个缺点是至少需要训练两个模型——奖励模型和LLM，这导致它的成本可能比实际所需的成本高。

直接偏好优化(direct preference optimization，DPO)是PPO的一种替代方案，它摒弃了基于强化学习的训练过程。DPO不再使用奖励模型来评判生成内容的质量，而是让LLM自己来完成这项工作。如图12-32所示，我们使用LLM的一个副本作为参考模型，评判参考模型和可训练模型在被接受的生成内容和被拒绝的生成内容的质量方面的偏移。

<div align="center">
    <img src="./picture/12-32.jpg">
    <p>图12-32：通过比较参考模型和可训练模型的输出，将LLM自身作为奖励模型</p>
</div>


通过在训练过程中计算这种偏移，我们可以通过跟踪参考模型和可训练模型之间的差异来优化被接受的生成内容相对于被拒绝的生成内容的似然概率。

为了计算这种偏移及其相关分数，我们需要从两个模型中提取被拒绝的生成内容和被接受的生成内容的对数概率。如图12-33所示，这个过程是在词元级别进行的，其中概率被组合起来，以计算参考模型和可训练模型之间的偏移。

图12-33：通过在词元级别获取生成内容的概率来计算分数。参考模型和可训练模型之间的概率偏移得到优化。被接受的生成内容也遵循相同的流程

我们可以使用这些分数优化可训练模型的参数，使其在生成被接受的内容时更有信心，在生成被拒绝的内容时更缺乏信心。与PPO相比，DPO在训练过程中更加稳定，准确度也更高。由于DPO的稳定性，我们将使用它对先前经过指令微调的模型进行偏好调优。

## 12.7 使用DPO进行偏好调优

当使用Hugging Face技术栈时，偏好调优与我们之前介绍的指令微调非常相似，只有一些细微的差异。我们仍将使用TinyLlama模型，但这次使用的是一个经过指令微调的版本。该模型首先使用全量微调进行训练，然后通过DPO进一步对齐。与最初的指令微调模型相比，这个模型是在更大的数据集上训练的。

在本节中，我们将演示如何使用DPO和基于奖励的数据集来进一步对齐此模型。

### 12.7.1 对齐数据的模板化

我们将使用一个数据集，其中每个提示词都包含一个被接受的生成内容和一个被拒绝的生成内容。这个数据集的一部分是由ChatGPT生成的，并对哪些输出应该被接受和哪些输出应该被拒绝进行了评分。

In [ ]:
from datasets import load_dataset
def format_prompt(example):
    """使用TinyLlama的<|user|>模板格式化提示词"""
    # 格式化答案
    system = "<|system|>\n" + example["system"] + "</s>\n"
    prompt = "<|user|>\n" + example["input"] + "</s>\n<|assistant|>\n"
    chosen = example["chosen"] + "</s>\n"
    rejected = example["rejected"] + "</s>\n"
    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }
# 格式化数据集并选择相对简短的答案
dpo_dataset = load_dataset(
    "argilla/distilabel-intel-orca-dpo-pairs", split="train"
, cache_dir="data")
dpo_dataset = dpo_dataset.filter(
    lambda r:
        r["status"] != "tie" and
        r["chosen_score"] >= 8 and
        not r["in_gsm8k_train"]
)
dpo_dataset = dpo_dataset.map(
    format_prompt, remove_columns=dpo_dataset.column_names
)
dpo_dataset

注意，我们应用了额外的过滤条件，将数据规模从原来的13000个样本进一步减少到大约6000个样本。

### 12.7.2 模型量化

我们加载基座模型和先前创建的LoRA。和之前一样，我们对模型进行量化以减少训练所需的显存。

In [6]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer
# 4位量化配置——QLoRA中的Q
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 使用4位精度模型加载
    bnb_4bit_quant_type="nf4",  # 量化类型
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算数据类型
    bnb_4bit_use_double_quant=True,  # 使用嵌套量化
)
# 合并LoRA和基座模型
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()
# 加载Llama分词器
model_name = "..\models\TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

<>:21: SyntaxWarning: invalid escape sequence '\m'
<>:21: SyntaxWarning: invalid escape sequence '\m'
C:\Users\admin\AppData\Local\Temp\ipykernel_19516\297398347.py:21: SyntaxWarning: invalid escape sequence '\m'
  model_name = "..\models\TinyLlama-1.1B-intermediate-step-1431k-3T"
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\peft\tuners\lora\bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


接下来，我们使用与之前相同的LoRA配置来执行DPO训练：

In [7]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# 准备LoRA配置
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA缩放
    lora_dropout=0.1,  # LoRA层的dropout
    r=64, # 秩
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # 目标层
     ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj", "down_proj"]
)
# 准备训练模型
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


### 12.7.3 训练配置

为简单起见，我们将使用与之前相同的训练参数，但有一点不同：我们不再运行一个完整的训练周期（这可能需要两小时）​，而仅运行200步用于演示。此外，我们添加了warmup_ratio参数，该参数在前10%的训练步数中将学习率从0增加到我们设置的learning_rate值。在训练开始阶段（预热期）保持较小的学习率，能够让模型先适应数据，然后应用更大的学习率，从而避免发生有害的发散。

In [8]:
from trl import DPOConfig
output_dir = "./results"
# 训练参数
training_arguments = DPOConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_steps=200,
    logging_steps=10,
    bf16=True,
    gradient_checkpointing=True,
    warmup_steps=20,
    beta=0.1,
    max_length=512
)

### 12.7.4 训练

我们已经准备好所有模型和参数，可以开始微调模型了：

In [9]:
from trl import DPOTrainer
# 创建DPO训练器
dpo_trainer = DPOTrainer(
    model,
    args=training_arguments,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)
# 使用DPO微调模型
dpo_trainer.train()
# 保存适配器
dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

Step,Training Loss
10,0.692609
20,0.671203
30,0.646853
40,0.595770
50,0.588981
60,0.604846
70,0.542826
80,0.484467
90,0.464604
100,0.537712


我们创建了第二个适配器。为了合并这两个适配器，我们通过迭代的方式将适配器与基座模型合并：

In [10]:
from peft import PeftModel
# 合并LoRA和基座模型
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)
sft_model = model.merge_and_unload()
# 合并DPO LoRA和SFT模型
dpo_model = PeftModel.from_pretrained(
    sft_model,
    "TinyLlama-1.1B-dpo-qlora",
    device_map="auto",
)
dpo_model = dpo_model.merge_and_unload()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

d:\study\python\AiAgent\图解大模型\.venv\Lib\site-packages\peft\peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lor

SFT和DPO相结合是一个很好的方法，可以先对模型进行微调以实现基本对话功能，然后根据人类偏好来调整其回答。但是，这也需要付出代价，因为我们需要执行两轮训练，并且可能需要在两个过程中调整参数。

自DPO发布以来，新的偏好对齐方法也不断出现。值得注意的是优势比偏好优化(odds ratio preference optimization，ORPO)，它将SFT和DPO合并为一个训练过程。ORPO不需要执行两轮训练，在允许使用QLoRA的同时，进一步简化了训练过程。

## 12.8 小结

在本章中，我们探讨了微调预训练LLM的不同步骤。我们通过使用低秩适配(LoRA)技术实现了参数高效微调(PEFT)。我们解释了如何通过量化（一种用于减少模型参数和适配器参数所占显存的技术）来扩展LoRA。

我们探讨的微调过程包含两个步骤。第一步，使用指令数据对预训练的LLM进行监督微调，这通常被称为指令调优。这使模型具备了类似对话的行为，并能够紧密地遵循指令。

第二步，基于对齐数据进行微调来进一步改进模型，这些对齐数据表示哪些类型的答案更受欢迎。这个过程被称为偏好调优，它将人类偏好提炼到之前经过指令微调的模型中。

总的来说，本章展示了微调预训练LLM的两个主要步骤，以及这些步骤如何带来更准确和更有价值的输出。